# 📚 MIDA507 — Big Data, Open Data et Data Science pour l'IDA
## Notebook Session 01 · Explorer le Big Data avec des Données Documentaires Africaines

---

| | |
|---|---|
| **Cours** | MIDA507 — Big Data, Open Data et Data Science pour l'IDA |
| **Programme** | Master MIDA — Université de Douala |
| **Session** | S01 — Fondements du Big Data et de la Data Science |
| **Outil** | Google Colaboratory (Python 3) |
| **Durée estimée** | 60 à 90 minutes |
| **Niveau requis** | Aucune expérience Python nécessaire — tout est commenté |

---

### 🎯 Ce que vous allez faire dans ce notebook

À la fin de ce notebook, vous serez capable de :

1. **Charger et explorer** un jeu de données documentaire africain avec Python
2. **Analyser les 7V du Big Data** sur un exemple concret tiré d'une bibliothèque universitaire camerounaise
3. **Identifier la valeur du rôle IDA** dans le pipeline de data science
4. **Produire des visualisations** simples pour communiquer la valeur d'un jeu de données

### 📦 Jeu de données utilisé

**Nom :** Données d'emprunt — Bibliothèque Centrale de l'Université de Ngaoundéré (fictif)
**Description :** 5 000 enregistrements simulant les emprunts de documents entre 2018 et 2023.
**Colonnes :** cote, titre_abrege, genre_document, date_emprunt, duree_jours, code_usager, filiere, niveau, region_origine, langue_document

> ⚠️ **Note pédagogique :** Ce jeu de données est entièrement fictif, généré pour les besoins du cours. Il reproduit cependant des réalités documentaires africaines authentiques : diversité des filières, origines géographiques variées, langues multiples.

---

---
## PARTIE 1 · Installation et Configuration

> **Aucune action requise de votre part.** Exécutez simplement chaque cellule dans l'ordre en cliquant sur le bouton ▶️ ou en appuyant sur `Shift + Entrée`.


In [ ]:
# ============================================================
# CELLULE 1.1 — Installation des bibliothèques nécessaires
# ============================================================
# Ces bibliothèques sont disponibles dans Google Colab.
# On les importe ci-dessous. Aucune installation nécessaire.

import pandas as pd          # Manipulation de données (tableaux)
import numpy as np           # Calculs numériques
import matplotlib.pyplot as plt  # Graphiques
import matplotlib.ticker as mticker
import seaborn as sns        # Graphiques statistiques avancés
import json                  # Manipulation de fichiers JSON
import warnings
from datetime import datetime, timedelta
import random

# Configuration de l'affichage
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid", palette="muted")

print("✅ Toutes les bibliothèques sont chargées avec succès.")
print(f"   pandas  version : {pd.__version__}")
print(f"   numpy   version : {np.__version__}")
print(f"   Python  OK")


---
## PARTIE 2 · Chargement du Jeu de Données

Le jeu de données simule les **emprunts enregistrés à la Bibliothèque Centrale de l'Université de Ngaoundéré** sur 5 ans (2018–2023). Il représente une situation réaliste pour une bibliothèque universitaire camerounaise de taille moyenne.


In [ ]:
# ============================================================
# CELLULE 2.1 — Génération du jeu de données fictif africain
# ============================================================
# Ce code génère 5 000 enregistrements simulant des emprunts
# de bibliothèque universitaire camerounaise.
# En production réelle, vous remplaceriez ce bloc par :
#   df = pd.read_csv("votre_fichier.csv")

random.seed(42)
np.random.seed(42)
N = 5000  # Nombre d'enregistrements

# --- Données de référence (réalistes pour le Cameroun) ---

genres = [
    "Thèse et mémoire", "Manuel universitaire", "Revue scientifique",
    "Rapport de recherche", "Ouvrage de référence",
    "Document administratif", "Atlas et cartographie", "Périodique"
]
poids_genres = [0.28, 0.30, 0.15, 0.10, 0.07, 0.05, 0.03, 0.02]

filieres = [
    "Droit", "Sciences économiques", "Lettres modernes",
    "Histoire", "Géographie", "Médecine", "Agronomie",
    "Informatique", "Sciences de l'éducation", "Sociologie"
]
poids_filieres = [0.18, 0.15, 0.12, 0.10, 0.08, 0.12, 0.08, 0.09, 0.05, 0.03]

niveaux = ["Licence 1", "Licence 2", "Licence 3",
           "Master 1", "Master 2", "Doctorat", "Enseignant-chercheur"]
poids_niveaux = [0.20, 0.18, 0.17, 0.15, 0.12, 0.10, 0.08]

regions = [
    "Adamaoua", "Centre", "Est", "Extrême-Nord",
    "Littoral", "Nord", "Nord-Ouest", "Ouest",
    "Sud", "Sud-Ouest"
]
poids_regions = [0.22, 0.18, 0.06, 0.12, 0.14, 0.10, 0.05, 0.07, 0.03, 0.03]

langues = ["Français", "Anglais", "Bilingue FR/EN", "Arabe", "Autres langues africaines"]
poids_langues = [0.62, 0.22, 0.10, 0.04, 0.02]

# --- Génération des données ---
date_debut = datetime(2018, 1, 1)
date_fin   = datetime(2023, 12, 31)
delta_total = (date_fin - date_debut).days

dates_emprunt = [
    date_debut + timedelta(days=random.randint(0, delta_total))
    for _ in range(N)
]

# Durée d'emprunt : distribution réaliste (pic à 7-14 jours)
durees = np.random.choice(
    [3, 7, 7, 7, 14, 14, 14, 21, 21, 30, 60],
    size=N,
    p=[0.05, 0.15, 0.15, 0.15, 0.10, 0.10, 0.10, 0.08, 0.05, 0.04, 0.03]
)

# Construction du DataFrame
df = pd.DataFrame({
    "cote":             [f"UNG-{str(i+1).zfill(5)}" for i in range(N)],
    "genre_document":   np.random.choice(genres, N, p=poids_genres),
    "date_emprunt":     dates_emprunt,
    "duree_jours":      durees,
    "code_usager":      [f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(N)],
    "filiere":          np.random.choice(filieres, N, p=poids_filieres),
    "niveau":           np.random.choice(niveaux, N, p=poids_niveaux),
    "region_origine":   np.random.choice(regions, N, p=poids_regions),
    "langue_document":  np.random.choice(langues, N, p=poids_langues),
})

# Ajout de l'année et du mois (utile pour les analyses)
df["annee"]       = df["date_emprunt"].dt.year
df["mois"]        = df["date_emprunt"].dt.month
df["mois_nom"]    = df["date_emprunt"].dt.strftime("%b")
df["date_emprunt"] = df["date_emprunt"].dt.strftime("%Y-%m-%d")

# Simulation de valeurs manquantes (réaliste : ~5% des champs filière)
idx_manquants = np.random.choice(df.index, size=int(N * 0.05), replace=False)
df.loc[idx_manquants, "filiere"] = np.nan

print(f"✅ Jeu de données généré : {len(df):,} enregistrements | {len(df.columns)} colonnes")
print(f"   Période couverte     : {df['date_emprunt'].min()} → {df['date_emprunt'].max()}")
print(f"   Usagers uniques      : {df['code_usager'].nunique():,}")
print(f"   Valeurs manquantes   : {df.isnull().sum().sum()} (filière uniquement)")


In [ ]:
# ============================================================
# CELLULE 2.2 — Aperçu du jeu de données
# ============================================================
# La méthode .head() affiche les 5 premières lignes.
# C'est toujours la PREMIÈRE chose à faire avec un nouveau jeu.

print("=" * 60)
print("APERÇU DES 5 PREMIÈRES LIGNES")
print("=" * 60)
df.head()


In [ ]:
# ============================================================
# CELLULE 2.3 — Informations générales sur le jeu de données
# ============================================================
# .info() donne : nombre de lignes, colonnes, types de données,
# et valeurs non-nulles. Essentiel pour l'audit qualité initial.

print("=" * 60)
print("INFORMATIONS GÉNÉRALES (df.info())")
print("=" * 60)
df.info()

print()
print("=" * 60)
print("STATISTIQUES DESCRIPTIVES (colonnes numériques)")
print("=" * 60)
df[["duree_jours"]].describe().round(1)


---
## PARTIE 3 · Analyse des 7V du Big Data sur notre Jeu de Données

> **Rappel théorique :** Les 7V (Volume, Vélocité, Variété, Véracité, Valeur, Variabilité, Visualisation) sont le cadre analytique fondamental pour évaluer tout jeu de données. Dans cette partie, vous allez mesurer concrètement chaque V sur notre jeu de bibliothèque.


### V1 — VOLUME : Quelle est la taille du jeu ?


In [ ]:
# ============================================================
# CELLULE 3.1 — V1 : VOLUME
# ============================================================
# Le volume ne désigne pas seulement le nombre de lignes,
# mais aussi la taille totale des données en mémoire.

nb_lignes    = len(df)
nb_colonnes  = len(df.columns)
taille_mo    = df.memory_usage(deep=True).sum() / 1024 / 1024
nb_usagers   = df["code_usager"].nunique()
nb_docs      = df["cote"].nunique()

print("╔══════════════════════════════════════════════════════════╗")
print("║              V1 — VOLUME                                ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Nombre d'enregistrements (emprunts)  : {nb_lignes:>8,}      ║")
print(f"║  Nombre de colonnes (variables)       : {nb_colonnes:>8}      ║")
print(f"║  Taille en mémoire                    : {taille_mo:>7.2f} Mo     ║")
print(f"║  Usagers uniques identifiés           : {nb_usagers:>8,}      ║")
print(f"║  Documents empruntés (cotes uniques)  : {nb_docs:>8,}      ║")
print("╚══════════════════════════════════════════════════════════╝")

print()
print("💡 INTERPRÉTATION IDA :")
print(f"   Ce jeu représente 5 ans de prêts pour une bibliothèque")
print(f"   universitaire camerounaise de taille moyenne.")
print(f"   Un système ILS (Koha, PMB) génère en réalité beaucoup")
print(f"   plus de données : logs de connexion, recherches, réservations.")
print(f"   Défi IDA → VOLUME : politique de rétention des données")
print(f"   (combien d'années conserver ? sous quel format ?)")


### V2 — VÉLOCITÉ : À quelle vitesse les données sont-elles produites ?


In [ ]:
# ============================================================
# CELLULE 3.2 — V2 : VÉLOCITÉ
# ============================================================
# On analyse la fréquence temporelle des enregistrements :
# combien d'emprunts par mois, par année ?

# Emprunts par année
emprunts_annee = df.groupby("annee").size().reset_index(name="nb_emprunts")

# Emprunts par mois (moyenne sur toutes les années)
emprunts_mois = df.groupby("mois").size().reset_index(name="nb_emprunts")
mois_labels = ["Jan","Fév","Mar","Avr","Mai","Jun",
               "Jul","Aoû","Sep","Oct","Nov","Déc"]
emprunts_mois["mois_label"] = mois_labels

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : évolution annuelle
axes[0].bar(emprunts_annee["annee"], emprunts_annee["nb_emprunts"],
            color="#E64A19", edgecolor="white", linewidth=0.5)
axes[0].set_title("Emprunts par année (2018–2023)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Année")
axes[0].set_ylabel("Nombre d'emprunts")
for i, row in emprunts_annee.iterrows():
    axes[0].text(row["annee"], row["nb_emprunts"] + 10,
                 str(row["nb_emprunts"]), ha="center", fontsize=9)

# Graphique 2 : saisonnalité mensuelle
axes[1].plot(emprunts_mois["mois_label"], emprunts_mois["nb_emprunts"],
             marker="o", color="#3E1500", linewidth=2, markersize=7)
axes[1].fill_between(range(12), emprunts_mois["nb_emprunts"],
                     alpha=0.15, color="#E64A19")
axes[1].set_title("Saisonnalité des emprunts (tous mois)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Mois")
axes[1].set_ylabel("Nombre d'emprunts")
axes[1].set_xticks(range(12))
axes[1].set_xticklabels(mois_labels)

plt.suptitle("V2 — VÉLOCITÉ : Rythme de production des données d'emprunt",
             fontsize=14, fontweight="bold", color="#3E1500", y=1.02)
plt.tight_layout()
plt.show()

# Calcul de la vitesse moyenne
jours_total = 365 * 5  # 5 ans
vitesse_jour = N / jours_total
print(f"
📊 Vitesse moyenne de production : {vitesse_jour:.1f} enregistrements / jour")
print(f"   Équivalent : {vitesse_jour * 30:.0f} enregistrements / mois")
print(f"
💡 DÉFI IDA → VÉLOCITÉ : En période d'examens (Oct–Nov, Jan–Fév),")
print(f"   le débit est 2x supérieur à la moyenne. Le système doit gérer")
print(f"   ces pics. L'IDA définit les règles d'archivage automatique.")


### V3 — VARIÉTÉ : Quels types de données coexistent ?


In [ ]:
# ============================================================
# CELLULE 3.3 — V3 : VARIÉTÉ
# ============================================================
# On analyse la diversité des types de documents, des filières,
# des niveaux et des langues dans notre jeu.

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

couleurs_ter = ["#BF360C","#E64A19","#FF7043","#FF8A65",
                "#FFAB91","#3E1500","#7B3F00","#A0522D"]

# --- Graphique 1 : Genres de documents ---
genre_counts = df["genre_document"].value_counts()
axes[0,0].barh(genre_counts.index, genre_counts.values,
               color=couleurs_ter[:len(genre_counts)], edgecolor="white")
axes[0,0].set_title("Genres de documents empruntés", fontweight="bold")
axes[0,0].set_xlabel("Nombre d'emprunts")
for i, v in enumerate(genre_counts.values):
    axes[0,0].text(v + 10, i, str(v), va="center", fontsize=9)

# --- Graphique 2 : Langues des documents ---
langue_counts = df["langue_document"].value_counts()
axes[0,1].pie(langue_counts.values,
              labels=langue_counts.index,
              autopct="%1.1f%%",
              colors=couleurs_ter[:len(langue_counts)],
              startangle=90)
axes[0,1].set_title("Langues des documents empruntés", fontweight="bold")

# --- Graphique 3 : Filières des emprunteurs ---
filiere_counts = df["filiere"].value_counts()
axes[1,0].barh(filiere_counts.index, filiere_counts.values,
               color="#558B2F", edgecolor="white")
axes[1,0].set_title("Filières des emprunteurs", fontweight="bold")
axes[1,0].set_xlabel("Nombre d'emprunts")

# --- Graphique 4 : Niveaux des emprunteurs ---
niveau_counts = df["niveau"].value_counts()
axes[1,1].bar(niveau_counts.index, niveau_counts.values,
              color="#1565C0", edgecolor="white")
axes[1,1].set_title("Niveaux d'étude des emprunteurs", fontweight="bold")
axes[1,1].set_ylabel("Nombre d'emprunts")
axes[1,1].tick_params(axis='x', rotation=30)

plt.suptitle("V3 — VARIÉTÉ : Diversité des types de données documentaires",
             fontsize=14, fontweight="bold", color="#3E1500", y=1.01)
plt.tight_layout()
plt.show()

print("
💡 DÉFI IDA → VARIÉTÉ :")
print(f"   Types distincts de documents : {df['genre_document'].nunique()}")
print(f"   Langues distinctes           : {df['langue_document'].nunique()}")
print(f"   Filières distinctes          : {df['filiere'].nunique()}")
print(f"   L'IDA doit concevoir un schéma de métadonnées qui décrit")
print(f"   toutes ces variétés de manière cohérente (Dublin Core,")
print(f"   puis DCAT pour la publication en open data — session 04).")


### V4 — VÉRACITÉ : Quelle est la qualité et la fiabilité des données ?


In [ ]:
# ============================================================
# CELLULE 3.4 — V4 : VÉRACITÉ
# ============================================================
# On mesure la qualité du jeu : valeurs manquantes, doublons,
# valeurs aberrantes. C'est le travail de base du data steward IDA.

print("═" * 60)
print("V4 — VÉRACITÉ : Audit de qualité initial")
print("═" * 60)

# 1. Valeurs manquantes
print("
📋 1. VALEURS MANQUANTES PAR COLONNE :")
manquants = df.isnull().sum()
taux_manquants = (manquants / len(df) * 100).round(2)
audit_manquants = pd.DataFrame({
    "Valeurs manquantes": manquants,
    "Taux (%)": taux_manquants
})
print(audit_manquants[audit_manquants["Valeurs manquantes"] > 0].to_string())

# 2. Doublons
nb_doublons = df.duplicated(subset=["code_usager","date_emprunt","cote"]).sum()
print(f"
📋 2. DOUBLONS (même usager, même cote, même date) : {nb_doublons}")

# 3. Valeurs aberrantes dans duree_jours
q1   = df["duree_jours"].quantile(0.25)
q3   = df["duree_jours"].quantile(0.75)
iqr  = q3 - q1
borne_sup = q3 + 1.5 * iqr
aberrants = df[df["duree_jours"] > borne_sup]

print(f"
📋 3. DURÉES D'EMPRUNT ABERRANTES (> {borne_sup:.0f} jours) :")
print(f"   Nombre de cas suspects : {len(aberrants)}")
print(f"   Durée max observée     : {df['duree_jours'].max()} jours")
print(f"   Durée médiane          : {df['duree_jours'].median():.0f} jours")

# Visualisation de la distribution des durées
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(df["duree_jours"], bins=20, color="#E64A19", edgecolor="white",
        alpha=0.8, label="Distribution normale")
ax.axvline(borne_sup, color="#1565C0", linestyle="--", linewidth=2,
           label=f"Seuil aberrant (>{borne_sup:.0f}j)")
ax.set_title("V4 — VÉRACITÉ : Distribution des durées d'emprunt",
             fontweight="bold", fontsize=13)
ax.set_xlabel("Durée d'emprunt (jours)")
ax.set_ylabel("Fréquence")
ax.legend()
plt.tight_layout()
plt.show()

print("
💡 DÉFI IDA → VÉRACITÉ :")
print("   Ces problèmes de qualité sont typiques d'une base de données")
print("   d'emprunt réelle. En session 05, nous apprendrons à les")
print("   corriger avec OpenRefine et Python. Le data steward IDA")
print("   définit les règles de qualité AVANT la publication.")


### V5 — VALEUR : Quelle connaissance peut-on extraire ?


In [ ]:
# ============================================================
# CELLULE 3.5 — V5 : VALEUR
# ============================================================
# On extrait des connaissances actionnables pour la direction
# de la bibliothèque : quoi acheter, quand ouvrir, qui sert-on ?

print("═" * 60)
print("V5 — VALEUR : Connaissances actionnables pour la direction")
print("═" * 60)

# ── Analyse 1 : Taux d'utilisation par filière ──
emprunts_filiere = (df.groupby("filiere")
                    .size()
                    .sort_values(ascending=False)
                    .reset_index(name="nb_emprunts"))

# ── Analyse 2 : Documents les plus empruntés par genre ──
top_genres = df.groupby("genre_document")["cote"].count().sort_values(ascending=False)

# ── Analyse 3 : Régions d'origine des usagers ──
top_regions = df.groupby("region_origine").size().sort_values(ascending=False)

# ── Analyse 4 : Durée moyenne par niveau ──
duree_niveau = (df.groupby("niveau")["duree_jours"]
                .mean()
                .sort_values(ascending=False)
                .round(1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Graphique 1 : Emprunts par filière
axes[0,0].bar(emprunts_filiere["filiere"],
              emprunts_filiere["nb_emprunts"],
              color=sns.color_palette("Oranges_r", len(emprunts_filiere)))
axes[0,0].set_title("Emprunts par filière", fontweight="bold")
axes[0,0].set_ylabel("Nombre d'emprunts")
axes[0,0].tick_params(axis='x', rotation=45)

# Graphique 2 : Top genres de documents
top_genres.plot(kind="pie", ax=axes[0,1], autopct="%1.0f%%",
                colors=sns.color_palette("RdBu_r", len(top_genres)),
                startangle=90)
axes[0,1].set_title("Répartition par genre de document", fontweight="bold")
axes[0,1].set_ylabel("")

# Graphique 3 : Régions d'origine
top_regions.plot(kind="bar", ax=axes[1,0], color="#558B2F", edgecolor="white")
axes[1,0].set_title("Régions d'origine des emprunteurs", fontweight="bold")
axes[1,0].set_ylabel("Nombre d'usagers")
axes[1,0].tick_params(axis='x', rotation=45)

# Graphique 4 : Durée moyenne par niveau
duree_niveau.plot(kind="barh", ax=axes[1,1], color="#1565C0", edgecolor="white")
axes[1,1].set_title("Durée moyenne d'emprunt par niveau d'étude", fontweight="bold")
axes[1,1].set_xlabel("Jours (moyenne)")

plt.suptitle("V5 — VALEUR : Tableaux de bord pour la direction de la bibliothèque",
             fontsize=14, fontweight="bold", color="#3E1500", y=1.01)
plt.tight_layout()
plt.show()

# Synthèse des insights
print("
🔍 CONNAISSANCES ACTIONNABLES POUR LA DIRECTION :")
print(f"   1. La filière la plus active : {emprunts_filiere.iloc[0]['filiere']}")
print(f"      → Priorité d'acquisition recommandée par l'IDA")
print(f"   2. Genre le plus emprunté    : {top_genres.index[0]}")
print(f"      → Vérifier si le stock est suffisant (politique de prêt)")
print(f"   3. Région la plus représentée: {top_regions.index[0]}")
print(f"      → Adapter les horaires d'ouverture aux contraintes régionales")
print(f"   4. Niveau qui garde le plus longtemps : {duree_niveau.index[0]}")
print(f"      ({duree_niveau.iloc[0]:.1f} jours) → Revoir la politique de durée ?")


### V6 & V7 — VARIABILITÉ et VISUALISATION


In [ ]:
# ============================================================
# CELLULE 3.6 — V6 : VARIABILITÉ + V7 : VISUALISATION
# ============================================================
# V6 : Le sens des données change selon le contexte
#      (période d'examens vs vacances, L1 vs Doctorat)
# V7 : Rendre les données lisibles pour les décideurs

# ── V6 : Variabilité — Comportement différent par période ──
df["periode"] = df["mois"].apply(lambda m:
    "Examens (Oct-Nov-Jan-Fév)" if m in [10,11,1,2]
    else ("Ramadan (Avr-Mai)" if m in [4,5]
    else "Période normale"))

variabilite = df.groupby(["periode","genre_document"]).size().unstack(fill_value=0)

# ── V7 : Tableau de bord de synthèse ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap : emprunts par mois et par genre
pivot_heatmap = df.pivot_table(
    index="genre_document", columns="annee",
    values="cote", aggfunc="count", fill_value=0
)
sns.heatmap(pivot_heatmap, ax=axes[0], cmap="YlOrRd",
            annot=True, fmt="d", linewidths=0.5,
            cbar_kws={"label": "Nb d'emprunts"})
axes[0].set_title("V7 — Évolution des emprunts par genre (2018-2023)
[Heatmap]",
                  fontweight="bold")
axes[0].set_xlabel("Année")
axes[0].set_ylabel("Genre de document")

# Boxplot : durée d'emprunt par période (V6 Variabilité)
data_box = [
    df[df["periode"] == p]["duree_jours"].values
    for p in df["periode"].unique()
]
bp = axes[1].boxplot(data_box, patch_artist=True,
                      labels=df["periode"].unique())
colors_box = ["#E64A19", "#1565C0", "#558B2F"]
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("V6 — Variabilité : Durée d'emprunt selon la période
[Boxplot]",
                  fontweight="bold")
axes[1].set_ylabel("Durée d'emprunt (jours)")
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle("V6 & V7 — Variabilité et Visualisation des données documentaires",
             fontsize=13, fontweight="bold", color="#3E1500", y=1.02)
plt.tight_layout()
plt.show()

print("
💡 DÉFI IDA → VARIABILITÉ :")
print("   Une consultation pendant les examens représente un besoin")
print("   urgent différent d'une consultation en période normale.")
print("   L'IDA doit contextualiser les données avec des métadonnées")
print("   temporelles pour que les analyses soient valides.")
print("
💡 DÉFI IDA → VISUALISATION :")
print("   Ces graphiques sont les livrables que la direction comprend.")
print("   L'IDA conçoit les indicateurs et les tableaux de bord —")
print("   compétence clé du data steward IDA en 2024.")


---
## PARTIE 4 · Positionner l'IDA dans le Pipeline de Data Science

> L'IDA intervient à **3 étapes clés** du pipeline : la collecte/évaluation, le nettoyage/documentation, et la communication/archivage. Cette partie illustre concrètement ces contributions.


In [ ]:
# ============================================================
# CELLULE 4.1 — Schéma du pipeline et rôle IDA
# ============================================================
# On affiche un schéma visuel du pipeline data science
# avec la contribution de l'IDA à chaque étape.

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 16)
ax.set_ylim(0, 5)
ax.axis("off")

etapes = [
    ("1. COLLECTE", 0.8, "#BF360C", "IDA ★★★
Localiser, évaluer
licences, droits"),
    ("2. NETTOYAGE", 3.2, "#E64A19", "IDA ★★★
Qualité, normes
traçabilité"),
    ("3. ANALYSE", 5.6, "#FF8A65", "IDA ★
Contextualiser
valider"),
    ("4. MODÉLISATION", 8.0, "#FFAB91", "IDA ★
Valider la logique
métier"),
    ("5. VISUALISATION", 10.4, "#FF7043", "IDA ★★
Concevoir KPIs
dashboards"),
    ("6. ARCHIVAGE", 12.8, "#BF360C", "IDA ★★★
DCAT, licences
préservation"),
]

for nom, x, couleur, role in etapes:
    # Boîte de l'étape
    rect = plt.Rectangle((x, 2.5), 2.2, 1.2,
                          facecolor=couleur, edgecolor="white",
                          linewidth=2, zorder=3)
    ax.add_patch(rect)
    ax.text(x + 1.1, 3.1, nom, ha="center", va="center",
            fontsize=8.5, fontweight="bold", color="white", zorder=4)

    # Boîte du rôle IDA
    ida_color = "#1A3A5C" if "★★★" in role else ("#2E7D32" if "★★" in role else "#9E9E9E")
    rect_ida = plt.Rectangle((x, 0.8), 2.2, 1.5,
                              facecolor=ida_color, edgecolor="white",
                              linewidth=1, alpha=0.85, zorder=3)
    ax.add_patch(rect_ida)
    ax.text(x + 1.1, 1.55, role, ha="center", va="center",
            fontsize=7.5, color="white", zorder=4)

    # Flèche entre les boîtes
    if x < 12.8:
        ax.annotate("", xy=(x + 2.4, 3.1), xytext=(x + 2.2, 3.1),
                    arrowprops=dict(arrowstyle="->", color="#555555", lw=2),
                    zorder=5)

# Légende
ax.text(8, 4.6, "PIPELINE DE DATA SCIENCE — Rôle du Data Steward IDA",
        ha="center", fontsize=14, fontweight="bold", color="#3E1500")
ax.text(0.5, 0.2, "Rôle IDA : ★★★ Central  ★★ Important  ★ Secondaire",
        fontsize=9, color="#555555", style="italic")

# Légende couleurs rôle
for label, color, x_pos in [("★★★ Rôle central IDA", "#1A3A5C", 8.5),
                              ("★★ Rôle important IDA", "#2E7D32", 10.5),
                              ("★ Rôle secondaire", "#9E9E9E", 12.5)]:
    ax.add_patch(plt.Rectangle((x_pos, 0.05), 0.3, 0.3,
                                facecolor=color, edgecolor="white"))
    ax.text(x_pos + 0.4, 0.2, label, fontsize=8, va="center", color="#333333")

plt.tight_layout()
plt.show()

print("
📊 SYNTHÈSE : L'IDA contribue DIRECTEMENT à 3 étapes sur 6 du pipeline")
print("   Collecte     (étape 1) : évaluation, droits, licences, sources")
print("   Nettoyage    (étape 2) : règles de qualité, OpenRefine, traçabilité")
print("   Archivage    (étape 6) : DCAT, licences, publication open data")
print()
print("   Dans nos prochaines sessions, nous pratiquerons chacune de ces étapes.")


In [ ]:
# ============================================================
# CELLULE 4.2 — Exemple concret : contribution IDA à l'étape 2
# ============================================================
# L'IDA définit les règles de qualité.
# Ici, on applique 3 règles simples sur notre jeu de données.

print("=" * 60)
print("CONTRIBUTION IDA — ÉTAPE 2 : Règles de qualité appliquées")
print("=" * 60)

df_propre = df.copy()

# ── Règle 1 : Durée d'emprunt valide (entre 1 et 90 jours) ──
masque_duree = (df_propre["duree_jours"] >= 1) & (df_propre["duree_jours"] <= 90)
nb_hors_duree = (~masque_duree).sum()
df_propre = df_propre[masque_duree]
print(f"
✅ Règle 1 — Durée valide (1-90 jours)")
print(f"   Enregistrements supprimés  : {nb_hors_duree}")
print(f"   Enregistrements conservés  : {len(df_propre):,}")

# ── Règle 2 : Filière manquante → étiquetée "Non renseignée" ──
nb_nans = df_propre["filiere"].isna().sum()
df_propre["filiere"] = df_propre["filiere"].fillna("Non renseignée")
print(f"
✅ Règle 2 — Filières manquantes renseignées")
print(f"   Valeurs renseignées        : {nb_nans}")
print(f"   Nouvelle valeur attribuée  : 'Non renseignée'")

# ── Règle 3 : Ajout d'un champ date_retour calculé ──
df_propre["date_retour"] = pd.to_datetime(df_propre["date_emprunt"])                            + pd.to_timedelta(df_propre["duree_jours"], unit="D")
df_propre["date_retour"] = df_propre["date_retour"].dt.strftime("%Y-%m-%d")
print(f"
✅ Règle 3 — Champ 'date_retour' calculé")
print(f"   Colonne ajoutée            : date_retour")

print(f"
{'─'*40}")
print(f"RÉSULTAT FINAL DU JEU NETTOYÉ :")
print(f"   Avant nettoyage : {len(df):,} lignes | {len(df.columns)} colonnes")
print(f"   Après nettoyage : {len(df_propre):,} lignes | {len(df_propre.columns)} colonnes")
print(f"
💡 Ces 3 règles simples constituent le début d'une politique")
print(f"   de qualité. En session 05, nous irons beaucoup plus loin")
print(f"   avec OpenRefine et les 6 dimensions du framework DAMA.")


---
## PARTIE 5 · Export du Livrable et Synthèse

> Cette partie produit votre livrable : un fichier CSV nettoyé et un rapport JSON d'inventaire, que vous utiliserez dans les sessions suivantes.


In [ ]:
# ============================================================
# CELLULE 5.1 — Export du jeu de données nettoyé
# ============================================================

# Export CSV
nom_fichier_csv = "MIDA507_S01_bibliotheque_ngaoundere_clean.csv"
df_propre.to_csv(nom_fichier_csv, index=False, encoding="utf-8-sig")

print(f"✅ Fichier CSV exporté : {nom_fichier_csv}")
print(f"   Lignes  : {len(df_propre):,}")
print(f"   Colonnes: {len(df_propre.columns)}")
print(f"   Encodage: UTF-8 avec BOM (compatible Excel africain)")


In [ ]:
# ============================================================
# CELLULE 5.2 — Rapport d'inventaire JSON (format IDA)
# ============================================================
# Ce rapport constitue le début d'une fiche de métadonnées.
# Il sera complété en session 04 avec le standard DCAT complet.

rapport = {
    "institution": "Bibliothèque Centrale — Université de Ngaoundéré",
    "pays": "Cameroun",
    "description": "Enregistrements d'emprunts documentaires 2018-2023. "
                   "Jeu de données fictif à des fins pédagogiques MIDA507.",
    "date_inventaire": datetime.now().strftime("%Y-%m-%d"),
    "volume": {
        "nb_enregistrements": int(len(df_propre)),
        "nb_colonnes": int(len(df_propre.columns)),
        "periode_couverte": "2018-01-01 / 2023-12-31"
    },
    "variete": {
        "genres_documents": df_propre["genre_document"].unique().tolist(),
        "langues_documents": df_propre["langue_document"].unique().tolist(),
        "filieres": df_propre["filiere"].unique().tolist(),
        "regions_origine": df_propre["region_origine"].unique().tolist()
    },
    "qualite": {
        "taux_completude_filiere": round(
            (df_propre["filiere"] != "Non renseignée").mean() * 100, 1),
        "duree_min_jours": int(df_propre["duree_jours"].min()),
        "duree_max_jours": int(df_propre["duree_jours"].max()),
        "duree_mediane_jours": float(df_propre["duree_jours"].median()),
        "nb_doublons": 0
    },
    "valeur_IDA": [
        "Identifier les filières prioritaires pour les acquisitions",
        "Adapter les horaires aux pics d'emprunt saisonniers",
        "Évaluer la représentation des langues africaines dans le fonds",
        "Argumenter les demandes budgétaires avec des données réelles"
    ],
    "prochaine_etape": "Session 02 — Évaluation FAIR et Data Management Plan"
}

nom_fichier_json = "MIDA507_S01_rapport_inventaire.json"
with open(nom_fichier_json, "w", encoding="utf-8") as f:
    json.dump(rapport, f, ensure_ascii=False, indent=2)

print(f"✅ Rapport d'inventaire exporté : {nom_fichier_json}")
print()
print("📄 Aperçu du rapport :")
print(json.dumps(rapport, ensure_ascii=False, indent=2)[:800] + "...")


In [ ]:
# ============================================================
# CELLULE 5.3 — Téléchargement des fichiers (Google Colab)
# ============================================================
# Dans Google Colab, cette cellule déclenche le téléchargement
# automatique des fichiers vers votre ordinateur.

try:
    from google.colab import files
    files.download(nom_fichier_csv)
    files.download(nom_fichier_json)
    print("✅ Téléchargement déclenché depuis Google Colab.")
except ImportError:
    # Si vous exécutez ce notebook localement (pas sur Colab)
    import os
    print("⚠️  Vous n'êtes pas sur Google Colab.")
    print(f"   Les fichiers sont sauvegardés dans : {os.getcwd()}")
    print(f"   - {nom_fichier_csv}")
    print(f"   - {nom_fichier_json}")


---
## PARTIE 6 · Exercices et Questions de Réflexion

> Complétez les zones **`[À COMPLÉTER]`** ci-dessous. Ces exercices constituent votre **livrable de participation** pour la session 01.


In [ ]:
# ============================================================
# EXERCICE 1 — Calculer le V5 (Valeur) pour votre institution
# ============================================================
# CONSIGNE : Modifiez le code ci-dessous pour répondre à
# la question suivante :
# "Quelle est la durée d'emprunt moyenne pour les THÈSES ET MÉMOIRES
#  uniquement ?"

# À COMPLÉTER : remplacez [TYPE_DOCUMENT] par le bon genre
type_recherche = "[TYPE_DOCUMENT]"  # ← Modifier ici

# Filtrer le jeu de données
df_filtre = df_propre[df_propre["genre_document"] == type_recherche]

# Calculer la durée moyenne
duree_moyenne = df_filtre["duree_jours"].mean()

print(f"Genre recherché        : {type_recherche}")
print(f"Nombre d'enregistrements : {len(df_filtre)}")
print(f"Durée moyenne d'emprunt  : {duree_moyenne:.1f} jours")

# ── Question de réflexion ──
print()
print("❓ QUESTION DE RÉFLEXION :")
print("   Si la durée moyenne des thèses est de 30 jours et celle des")
print("   manuels de 7 jours, quelle politique de prêt recommanderiez-vous")
print("   à la direction de la bibliothèque ? Rédigez 3 lignes ci-dessous.")
print()
print("   VOTRE RÉPONSE :")
print("   [À RÉDIGER]")


In [ ]:
# ============================================================
# EXERCICE 2 — Analyser la représentation régionale
# ============================================================
# CONSIGNE : Calculez et affichez le TOP 3 des régions
# d'origine des emprunteurs en terme de durée moyenne d'emprunt.
# (Pas en nombre d'emprunts — en durée MOYENNE)

# À COMPLÉTER
duree_par_region = (df_propre
    .groupby("[NOM_COLONNE]")  # ← Remplacer par le nom de la colonne région
    ["[NOM_COLONNE_DUREE]"]    # ← Remplacer par le nom de la colonne durée
    .mean()
    .sort_values(ascending=False)
    .head(3)
    .round(1)
)

print("TOP 3 des régions par durée moyenne d'emprunt :")
print(duree_par_region)

print()
print("❓ QUESTION DE RÉFLEXION :")
print("   Que révèle une durée d'emprunt plus longue dans certaines régions ?")
print("   Lien possible avec la distance campus / résidence de l'étudiant ?")
print("   Recommandation IDA : [À RÉDIGER]")


In [ ]:
# ============================================================
# EXERCICE 3 — Synthèse des 7V pour votre institution réelle
# ============================================================
# CONSIGNE : Complétez le tableau ci-dessous en appliquant
# les 7V à UN jeu de données de VOTRE institution réelle.
# (Si vous n'avez pas accès à des données réelles, utilisez
# une institution africaine fictive que vous connaissez bien.)

synthese_7V = {
    "Institution": "[NOM DE VOTRE INSTITUTION]",     # ← À compléter
    "Jeu de données choisi": "[DESCRIPTION]",         # ← À compléter
    "V1_Volume": {
        "estimation": "[NOMBRE APPROXIMATIF D'ENREGISTREMENTS]",
        "defi_IDA": "[QUEL PROBLÈME DE STOCKAGE/RÉTENTION ?]"
    },
    "V2_Velocite": {
        "frequence_production": "[QUOTIDIENNE / HEBDOMADAIRE / MENSUELLE ?]",
        "defi_IDA": "[QUEL PROBLÈME D'ARCHIVAGE EN TEMPS RÉEL ?]"
    },
    "V3_Variete": {
        "types_donnees": ["[TYPE 1]", "[TYPE 2]", "[TYPE 3]"],
        "defi_IDA": "[QUEL PROBLÈME D'INTEROPÉRABILITÉ ?]"
    },
    "V4_Veracite": {
        "probleme_qualite_principal": "[DÉCRIVEZ UN PROBLÈME CONCRET]",
        "defi_IDA": "[QUELLE RÈGLE DE QUALITÉ PROPOSEZ-VOUS ?]"
    },
    "V5_Valeur": {
        "question_strategique": "[QUELLE DÉCISION CES DONNÉES POURRAIENT-ELLES ÉCLAIRER ?]",
        "defi_IDA": "[COMMENT VALORISER CES DONNÉES ?]"
    },
    "V6_Variabilite": {
        "contexte_change": "[DANS QUEL CAS LE SENS DES DONNÉES CHANGE-T-IL ?]",
        "defi_IDA": "[QUELLES MÉTADONNÉES CONTEXTUELLES AJOUTER ?]"
    },
    "V7_Visualisation": {
        "indicateur_cle": "[QUEL KPI PRÉSENTERIEZ-VOUS À VOTRE DIRECTEUR ?]",
        "defi_IDA": "[QUEL OUTIL DE VISUALISATION UTILISERIEZ-VOUS ?]"
    }
}

# Affichage formaté
print("╔══════════════════════════════════════════════════════════╗")
print("║     ANALYSE 7V — MON INSTITUTION                        ║")
print("╠══════════════════════════════════════════════════════════╣")
for v_key, v_val in synthese_7V.items():
    if isinstance(v_val, dict):
        print(f"
  {v_key.upper()} :")
        for sous_cle, sous_val in v_val.items():
            if isinstance(sous_val, list):
                print(f"    {sous_cle} : {', '.join(sous_val)}")
            else:
                print(f"    {sous_cle} : {sous_val}")
    else:
        print(f"
  {v_key} : {v_val}")

# Sauvegarde de la synthèse
nom_synthese = "MIDA507_S01_Synthese_7V_MonInstitution.json"
with open(nom_synthese, "w", encoding="utf-8") as f:
    json.dump(synthese_7V, f, ensure_ascii=False, indent=2)

print(f"
✅ Synthèse sauvegardée : {nom_synthese}")
print("   → À partager avec l'enseignant comme livrable de session 01")


---
## ✅ Bilan du Notebook S01 — Ce que vous avez accompli

| Compétence travaillée | Statut |
|---|---|
| Charger et explorer un jeu de données pandas | ✅ |
| Calculer et interpréter les 7V du big data | ✅ |
| Identifier les contributions IDA dans le pipeline data science | ✅ |
| Produire des visualisations matplotlib/seaborn | ✅ |
| Exporter un CSV nettoyé et un rapport JSON | ✅ |
| Appliquer les 7V à votre institution réelle (exercice 3) | 🟡 À compléter |

---

## 🔗 Lien avec les sessions suivantes

| Session | Ce que vous utiliserez de S01 |
|---|---|
| **S02 — Cycle de vie et FAIR** | Le CSV exporté sera évalué selon les principes FAIR |
| **S04 — Métadonnées DCAT** | Le rapport JSON sera converti en fiche DCAT complète |
| **S05 — Qualité et OpenRefine** | Le jeu dégradé sera nettoyé avec OpenRefine |
| **S06 — Publication open data** | Pipeline complet appliqué à votre jeu personnel |

---

## 📎 Fichiers produits par ce notebook

1. `MIDA507_S01_bibliotheque_ngaoundere_clean.csv` — Jeu de données nettoyé (5 000 lignes)
2. `MIDA507_S01_rapport_inventaire.json` — Rapport d'inventaire IDA
3. `MIDA507_S01_Synthese_7V_MonInstitution.json` — Votre synthèse 7V (livrable de session)

---

*Notebook MIDA507 S01 · Master MIDA — Université de Douala · Cours Big Data, Open Data et Data Science pour l'IDA*
